# CNN图像分类


由 [Vision Learning Hub](https://charlie-sys.github.io/vision-learning-hub/) 提供。
在 Colab 登录自己的Google账号，按从上到下的顺序运行；修改前可“在云端硬盘中保存副本”。
若需要GPU，在“运行时 → 更改运行时类型”中选择可用GPU。免费GPU不保证随时可用，断线后临时文件可能丢失，请下载结果。
本Notebook的结果属于你自己的运行，不应把教程实测数值直接当作本次成绩。


## 1. 加载经过版本和SHA256校验的课程代码
首次需要网络。下载源为本课程公开GitHub仓库；可在运行前查看链接内容。若依赖缺失，请按实验网页安装匹配的torch/torchvision。


In [ ]:
from pathlib import Path
import hashlib, urllib.request, importlib.util, argparse, json
source_url = 'https://raw.githubusercontent.com/charlie-sys/vision-learning-hub/0a4d92e45e9b29e8dea73b85aa5889cc563afb09/examples/course_labs.py'
folder = Path('vision_learning_runtime')
folder.mkdir(exist_ok=True)
source = folder / 'course_labs.py'
urllib.request.urlretrieve(source_url, source)
assert hashlib.sha256(source.read_bytes()).hexdigest() == 'f52709931d60aacbe5d28788690dc4498f2e4c82c23943bfffd8013eec5234e4', 'Source checksum differs; stop and verify the source.'
spec = importlib.util.spec_from_file_location('course_labs', source)
labs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(labs)
print('Verified course source: 0a4d92e45e9b29e8dea73b85aa5889cc563afb09')


## 2. 配置
先保留默认设置跑通。CPU较慢时减少样本和轮数；synthetic只用于流程检查，不能替代FashionMNIST或真实DINO结果。


In [ ]:
import torch
from IPython.display import display, Image, SVG
args = argparse.Namespace(
    mode='cnn', dataset='fashion', seed=7,
    data_dir=str(folder / 'datasets'), output=str(folder / 'runs-cnn'),
    train_size=2048, val_size=512,
    epochs=3, batch_size=64, lr=0.001,
    device='cuda' if torch.cuda.is_available() else 'cpu', resume=None,
    latent_dim=16, beta=1., steps=200, encoder='dinov2',
    hub_ref=labs.DINO_REF, hub_dir=None, random_baseline=True, evaluate_test=False,
)
Path(args.output).mkdir(parents=True, exist_ok=True)
labs.seed_all(args.seed)
print('torch:', torch.__version__, 'device:', args.device)
print(vars(args))


## 3. 数据划分
仅使用官方训练数据中的不重叠训练/验证子集；不读取官方测试集。DINO的特征标准化仅用训练特征统计。


In [ ]:
train, val, test, metadata = labs.data_splits(args)
x, y = next(iter(labs.loader(train, args)))
print('batch:', tuple(x.shape), tuple(y.shape))
print('split metadata:', metadata)


## 4. 训练与验证
阅读网页中对应损失与流程说明，再运行。最佳模型根据验证集选择，不能把验证结果当作测试成绩。DINO只训练线性头，不训练预训练骨干。


In [ ]:
history, result = labs.train_cnn(args, train, val, test)
print(json.dumps(result, indent=2))
metadata.update(config=vars(args), results=result, history=history, torch=str(torch.__version__))
(Path(args.output) / 'metrics.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
labs.write_curve(history, Path(args.output) / 'curves.svg')


## 5. 查看曲线、图像和检查点
图像显示当前运行结果。小模型可能产生模糊或破碎样本，请记录失败情况，不仅展示最好一张。


In [ ]:
display(SVG(filename=str(Path(args.output) / 'curves.svg')))
for image in sorted(Path(args.output).glob('*.png')):
    print(image.name)
    display(Image(filename=str(image)))
print('Artifacts:', [p.name for p in Path(args.output).iterdir()])


## 6. 打包结果
下载ZIP前确认内容；文件含训练权重和配置，不包含原始数据集。Colab左侧文件面板可以下载下方生成的ZIP。


In [ ]:
import shutil
archive = shutil.make_archive('results_cnn', 'zip', args.output)
print('Download this file using the notebook file panel:', archive)


## 练习与验收
比较学习率0.001与0.01；解释混淆矩阵中最常见的错误。

修改seed为17、27并更换输出目录，报告均值和样本标准差。[完整教学与排错](https://charlie-sys.github.io/vision-learning-hub/practice.html)。
